In [2]:
import os, math, torch, types, random
import numpy as np
from PIL import Image
from diffusers import SanaPipeline
from torch.nn import Identity

# --------------------------------------------
# 0) Set seed for reproducibility
# --------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"✔ Seed set to {SEED}")


# --------------------------------------------
# 1) Load pipeline
# --------------------------------------------
pipe = SanaPipeline.from_pretrained(
    "Efficient-Large-Model/Sana_1600M_1024px_diffusers",
    variant="fp16",
    torch_dtype=torch.float16,
).to("cuda")

pipe.vae.to(torch.bfloat16)
pipe.text_encoder.to(torch.bfloat16)

print("✔ Pipeline loaded")


# --------------------------------------------
# 0.1) Install forward with identities
# --------------------------------------------
def install_forward_with_identities(block):
    """Add identity layers after gating (from train_probes_online.py)"""
    block.identity_after_attn = Identity()
    block.identity_after_ff = Identity()

    def forward2(self, hidden_states, attention_mask=None, encoder_hidden_states=None,
                 encoder_attention_mask=None, timestep=None, height=None, width=None):
        batch_size = hidden_states.shape[0]
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
            self.scale_shift_table[None] + timestep.reshape(batch_size, 6, -1)).chunk(6, dim=1)

        # Self-Attention
        norm_hidden_states = self.norm1(hidden_states)
        norm_hidden_states = norm_hidden_states * (1 + scale_msa) + shift_msa
        norm_hidden_states = norm_hidden_states.to(hidden_states.dtype)
        attn_output = self.attn1(norm_hidden_states)
        hidden_states = hidden_states + self.identity_after_attn(gate_msa * attn_output)

        # Cross-Attention
        if self.attn2 is not None:
            attn_output = self.attn2(hidden_states, encoder_hidden_states=encoder_hidden_states,
                                    attention_mask=encoder_attention_mask)
            hidden_states = hidden_states + attn_output

        # Feed-forward
        norm_hidden_states = self.norm2(hidden_states)
        norm_hidden_states = norm_hidden_states * (1 + scale_mlp) + shift_mlp
        norm_hidden_states = norm_hidden_states.unflatten(1, (height, width)).permute(0, 3, 1, 2)
        ff_output = self.ff(norm_hidden_states)
        ff_output = ff_output.flatten(2, 3).permute(0, 2, 1)
        hidden_states = hidden_states + self.identity_after_ff(gate_mlp * ff_output)

        return hidden_states

    block.forward = types.MethodType(forward2, block)

for block in pipe.transformer.transformer_blocks:
    install_forward_with_identities(block)

print("✔ Installed identity layers")

prompt = "A house in the bottom-left, children in the bottom-right, the sun in the top-left, and clouds in the top-right."
run_id = "exp_correct_norm_out"
root_dir = f"exp1_results/{run_id}"
os.makedirs(root_dir, exist_ok=True)

with open(os.path.join(root_dir, "prompt.txt"), "w") as f:
    f.write(prompt)


# =====================================================
# 1) Capture timestep from transformer forward (like train_probes_online.py)
# =====================================================
real_timestep = None  # track real timestep value
step_idx = {"t": -1}  # track diffusion step index

def transformer_forward_pre_hook(mod, args, kwargs=None):
    """Capture timestep from transformer forward call (before patch_embed fires)"""
    global real_timestep
    
    # Get timestep from kwargs (Sana pipeline passes it as keyword argument)
    if not kwargs or 'timestep' not in kwargs:
        raise RuntimeError(
            f"Could not find 'timestep' in kwargs. "
            f"Args: {len(args)}, kwargs: {list(kwargs.keys()) if kwargs else 'None'}. "
            f"Make sure with_kwargs=True is set when registering this hook."
        )
    
    timestep = kwargs['timestep']
    print(f"timestep: {timestep}")
    # Extract scalar value
    if isinstance(timestep, torch.Tensor):
        real_timestep = int(timestep[0].item()) if timestep.dim() > 0 else int(timestep.item())
    else:
        real_timestep = int(timestep)
    print(f"real_timestep: {real_timestep}")
    # Increment step index
    step_idx["t"] += 1

pipe.transformer.register_forward_pre_hook(transformer_forward_pre_hook, with_kwargs=True)


# =====================================================
# 2) Capture embedded timestep for norm_out transformation
# =====================================================
stored_emb = {}     # stores embedded timestep vectors

def time_embed_hook(module, inp, out):
    """
    Capture embedded_timestep for use in apply_transform (logit lens).
    out = (linear_output, embedded_timestep)
    """
    if real_timestep is not None:
        embedded_timestep = out[1].detach().clone()
        stored_emb[real_timestep] = embedded_timestep

pipe.transformer.time_embed.register_forward_hook(time_embed_hook)


# =====================================================
# 3) Helper: apply correct final SANA transformation
# =====================================================
def apply_transform(lat, timestep):
    """
    Apply the SAME:
        norm_out(lat, embedded_timestep[t], scale_shift_table)
        proj_out(lat)

    This is the ONLY mathematically correct way.
    """
    emb = stored_emb[timestep]
    sst = pipe.transformer.scale_shift_table

    # Apply AdaLayerNorm
    lat = pipe.transformer.norm_out(lat, emb, sst)

    # Apply final projection 2240 → 32
    lat = pipe.transformer.proj_out(lat)

    return lat


# =====================================================
# 4) Decode and save intermediate activations
# =====================================================
@torch.no_grad()
def decode_and_save(lat, tag, block=None, apply_final_transform=True):
    """
    Decode intermediate activations to images.
    Uses global real_timestep and step_idx for proper tracking.
    """
    if real_timestep is None:
        raise ValueError("Timestep not captured yet")
        return  # Skip if timestep not captured yet

    # Flatten (if tuple)
    if isinstance(lat, tuple):
        lat = lat[0]

    # Case 1: (B, S, C)
    if lat.ndim == 3:
        B, S, C = lat.shape
        H = W = int(math.isqrt(S))

        if C == 2240 and apply_final_transform:
            lat = apply_transform(lat, real_timestep)  # Use real_timestep!

        lat = lat.transpose(1, 2).reshape(B, 32, H, W)

    # Case 2: (B, C, H, W)
    else:
        B, C, H, W = lat.shape

        if C == 2240 and apply_final_transform:
            lat = lat.permute(0, 2, 3, 1).reshape(B, H * W, 2240)
            lat = apply_transform(lat, real_timestep)  # Use real_timestep!
            lat = lat.transpose(1, 2).reshape(B, 32, H, W)

    # Decode with VAE
    lat = lat.float() / pipe.vae.config.scaling_factor
    lat = lat.to(pipe.vae.decoder.conv_in.weight.dtype)
    rgb = pipe.vae.decode(lat).sample.float()

    img = ((rgb[0] * 0.5 + 0.5)
           .clamp(0, 1) * 255).permute(1, 2, 0).cpu().to(torch.uint8).numpy()

    # Save with both step_idx and real_timestep in folder name
    sub = os.path.join(root_dir, f"step_{step_idx['t']:02d}_t{real_timestep}")
    os.makedirs(sub, exist_ok=True)
    fn = f"{tag}.png" if block is None else f"block{block:02d}_{tag}.png"
    Image.fromarray(img).save(os.path.join(sub, fn))



# =====================================================
# 5) Register hooks on identity layers (after gating)
# =====================================================
def make_post_hook(tag, idx):
    def hook(m, inp, out):
        decode_and_save(out, tag, idx)  # Removed step parameter
    return hook


for blk_id, blk in enumerate(pipe.transformer.transformer_blocks):
    blk.identity_after_attn.register_forward_hook(make_post_hook("self_attn_after_gate", blk_id))
    blk.identity_after_ff.register_forward_hook(make_post_hook("mix_ffn_after_gate", blk_id))


# =====================================================
# 6) Save the final transformer output (no transform needed)
# =====================================================
pipe.transformer.register_forward_hook(
    lambda _m, _inp, out:
        decode_and_save(
            out[0] if isinstance(out, tuple) else out,
            "transformer_out",
            apply_final_transform=False  # Already has norm_out + proj_out applied
        )
)


# =====================================================
# 7) Run diffusion
# =====================================================
num_steps = 20
pipe.scheduler.set_timesteps(num_steps)

generator = torch.Generator(device="cuda").manual_seed(SEED)

with torch.inference_mode():
    images = pipe(
        prompt=prompt,
        guidance_scale=5.0,
        num_inference_steps=num_steps,
        generator=generator,
    ).images

images[0].save(os.path.join(root_dir, "final.png"))
print(f"✔ All activations saved under “{root_dir}/”")


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✔ Pipeline loaded
✔ Installed identity layers


  0%|          | 0/20 [00:00<?, ?it/s]

timestep: tensor([999., 999.], device='cuda:0')
real_timestep: 999
timestep: tensor([982., 982.], device='cuda:0')
real_timestep: 982
timestep: tensor([963., 963.], device='cuda:0')
real_timestep: 963
timestep: tensor([944., 944.], device='cuda:0')
real_timestep: 944
timestep: tensor([922., 922.], device='cuda:0')
real_timestep: 922
timestep: tensor([899., 899.], device='cuda:0')
real_timestep: 899
timestep: tensor([874., 874.], device='cuda:0')
real_timestep: 874
timestep: tensor([847., 847.], device='cuda:0')
real_timestep: 847
timestep: tensor([817., 817.], device='cuda:0')
real_timestep: 817
timestep: tensor([785., 785.], device='cuda:0')
real_timestep: 785
timestep: tensor([749., 749.], device='cuda:0')
real_timestep: 749
timestep: tensor([710., 710.], device='cuda:0')
real_timestep: 710
timestep: tensor([666., 666.], device='cuda:0')
real_timestep: 666
timestep: tensor([617., 617.], device='cuda:0')
real_timestep: 617
timestep: tensor([562., 562.], device='cuda:0')
real_timestep:

KeyboardInterrupt: 